# Estudo de Caso 01: Motor de Física, Geometria de Estádios e Tabelas de Parede

> **HaxBall RL Suite** - Simulação determinística de física 2D a 60 FPS e suporte a mapas oficiais `.hbs`.

Neste notebook, exploramos:
1. O parser oficial de estádios JSON5 (`.hbs`) com propriedades de restituição, segmentos e traves.
2. A comparação em escala entre os mapas oficiais: **Futsal 2v2**, **Futsal 3v3 (7899)**, **Futsal 5v5 (9362)** e **Micro 1v1**.
3. A física de reflexão elástica nas paredes (*tabelas* com $bCoef = 1.25$).
4. A condução inelástica de futsal ($bCoef = 0.0$) e o sistema de *kick rate-limit*.


In [ ]:
import os
import sys
import math
import numpy as np
import matplotlib.pyplot as plt

# Adiciona o diretório raiz ao path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from haxball.core.vector import Vec2
from haxball.core.constants import Team, GameState
from haxball.core.stadium import Stadium
from haxball.core.game import HaxBallGame
from haxball.core.physics_engine import PhysicsEngine

print('Módulos HaxBall carregados com sucesso!')


## 1. Carregando e Inspecionando os Estádios Oficiais da Comunidade

Os estádios do HaxBall são especificados em formato JSON5. Nosso parser resolve herança de *traits*, segmentos curvos e parâmetros de colisão.


In [ ]:
MAP_DIR = os.path.join('..', 'haxball', 'maps')
maps_info = [
    ('Futsal 2v2 Arena', 'futsal_2v2.hbs'),
    ('Futsal 3v3 GLH (Oficial 7899)', 'futsal_3v3.hbs'),
    ('Futsal 5v5 GLH (Oficial 9362)', 'futsal_5v5.hbs'),
    ('Micro 1v1 Arena', 'micro_1v1.hbs'),
    ('Big Stadium Oficial', 'big_stadium.hbs'),
    ('Classic Stadium', 'classic.hbs')
]

stadiums = {}
for name, fname in maps_info:
    path = os.path.join(MAP_DIR, fname)
    stad = Stadium.load_from_file(path)
    stadiums[name] = stad
    print(f'✓ {name:<32} | Dimensões: {stad.width*2}x{stad.height*2} px | Segmentos: {len(stad.segments):<3} | Gols: {len(stad.goals)}')

# Estatísticas comparativas de escala
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (name, fname) in enumerate(maps_info):
    ax = axes[i]
    stad = stadiums[name]
    # Desenha o piso da quadra
    ax.fill([-stad.bg_width, stad.bg_width, stad.bg_width, -stad.bg_width],
            [-stad.bg_height, -stad.bg_height, stad.bg_height, stad.bg_height],
            color='#455A64', alpha=0.85, label='Quadra')
    # Linha central e kickoff
    ax.axvline(0, color='white', linestyle='--', alpha=0.7)
    circle = plt.Circle((0, 0), stad.bg_kickoff_radius, color='white', fill=False, linewidth=1.5)
    ax.add_patch(circle)
    
    # Gols
    for g in stad.goals:
        c = 'red' if g.team == Team.RED else 'blue'
        ax.plot([g.p0.x, g.p1.x], [g.p0.y, g.p1.y], color=c, linewidth=4, label=f'Gol {g.team.name}')
        
    ax.set_title(f'{name}\n({stad.width*2}x{stad.height*2} px)', fontsize=11, fontweight='bold')
    ax.set_xlim(-stad.width - 40, stad.width + 40)
    ax.set_ylim(-stad.height - 40, stad.height + 40)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


## 2. Simulação de Trajetória e Dinâmica de Tabelas de Parede

Nas partidas competitivas de HaxBall (*tryhard*), os jogadores utilizam o rebote nas paredes para superar adversários.
A lei de reflexão elástica com restituição $e = bCoef_1 \times bCoef_2$ dita o vetor após colisão com o plano normal $\hat{n}$:
$$\vec{v}' = \vec{v} - (1 + e)(\vec{v} \cdot \hat{n})\hat{n}$$
Em quadras de futsal como a **7899**, as paredes têm $bCoef = 1.25$ para chutes explosivos de tabela.


In [ ]:
stad = stadiums['Futsal 3v3 GLH (Oficial 7899)']
game = HaxBallGame(stadium=stad, red_players_count=1, blue_players_count=1)

# Posiciona a bola no centro e aplica um chute em direção à parede superior em ângulo
game.ball.pos = Vec2(0.0, 0.0)
game.ball.speed = Vec2(8.0, 9.0)  # Velocidade inicial oblíqua

traj_x = [game.ball.pos.x]
traj_y = [game.ball.pos.y]

# Simula 80 ticks de física pura
for tick in range(80):
    game.physics.step({})
    traj_x.append(game.ball.pos.x)
    traj_y.append(game.ball.pos.y)

plt.figure(figsize=(10, 5))
# Desenha limites da quadra
plt.plot([-stad.bg_width, stad.bg_width, stad.bg_width, -stad.bg_width, -stad.bg_width],
         [-stad.bg_height, -stad.bg_height, stad.bg_height, stad.bg_height, -stad.bg_height],
         'k-', linewidth=2, label='Paredes da Quadra')

plt.plot(traj_x, traj_y, 'o-', color='#FF5722', markersize=3, linewidth=2, label='Trajetória com Tabela')
plt.plot(traj_x[0], traj_y[0], 'go', markersize=10, label='Disparo Inicial')
plt.plot(traj_x[-1], traj_y[-1], 'rs', markersize=10, label='Posição Final')

plt.title('Simulação de Chute em Tabela (Wall Rebound) no Futsal 3v3', fontsize=12, fontweight='bold')
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f'Velocidade inicial: {Vec2(8.0, 9.0).length():.2f} px/tick')
print(f'Velocidade final após tabela e amortecimento: {game.ball.speed.length():.2f} px/tick')


## 3. Condução Inelástica de Futsal ($bCoef = 0.0$)

Em mapas de futsal, o jogador possui `bCoef = 0.0`. Quando o jogador encosta na bola sem acionar o chute, o impacto é absorvido, permitindo conduzir a bola rente ao corpo (*dribble tapping*).


In [ ]:
stad = stadiums['Futsal 2v2 Arena']
game = HaxBallGame(stadium=stad, red_players_count=1, blue_players_count=1)

p = game.players[0]
p.pos = Vec2(-50.0, 0.0)
game.ball.pos = Vec2(0.0, 0.0)
game.ball.speed = Vec2(0.0, 0.0)

# Jogador acelera em direção à bola SEM chutar (apenas empurrando)
speeds = []
ball_speeds = []
for _ in range(60):
    game.step({p.player_id: (1.0, 0.0, False)})
    speeds.append(p.speed.x)
    ball_speeds.append(game.ball.speed.x)

plt.figure(figsize=(10, 4))
plt.plot(speeds, label='Velocidade Jogador (X)', linewidth=2)
plt.plot(ball_speeds, label='Velocidade Bola (X) - Condução Colada', linewidth=2, linestyle='--')
plt.title('Dinâmica de Condução Colada (Inelástica) no Futsal', fontsize=12, fontweight='bold')
plt.xlabel('Tick')
plt.ylabel('Velocidade (px/tick)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('✓ A bola é acelerada de forma contínua sem repulsão descontrolada!')
